# SEMICON AdaIR Colab Training and Testing

This notebook trains the AdaIR wafer restoration model on `.npy` NoisyLR/GT pairs, saves the best validation checkpoint, validates PSNR/SSIM/LPIPS, and exports restored `.npy` outputs for testing.


## 1. Runtime Setup

Use **Runtime > Change runtime type > GPU** before running the notebook.


In [ ]:
!nvidia-smi
!python --version


## 2. Install Dependencies


In [ ]:
!pip -q install pyyaml lpips scikit-image tqdm matplotlib


## 3. Prepare Project Files

These cells recreate the project code inside Colab. Re-run them whenever you restart the runtime.


In [ ]:
!mkdir -p datasets models configs checkpoints results reports utils
!touch datasets/__init__.py models/__init__.py utils/__init__.py


In [ ]:
%%writefile datasets/semicon_dataset.py
import os
import glob
import numpy as np
import torch
from torch.utils.data import Dataset
import random

class SemiconDataset(Dataset):
    def __init__(self, noisy_dir, gt_dir, patch_size=128, phase='train', seed=42):
        self.noisy_paths = sorted(glob.glob(os.path.join(noisy_dir, '*.npy')))
        self.gt_paths = sorted(glob.glob(os.path.join(gt_dir, '*.npy')))
        self.patch_size = patch_size
        self.phase = phase
        random.seed(seed)
        torch.manual_seed(seed)
        np.random.seed(seed)
        
        # Verify dataset length
        assert len(self.noisy_paths) == len(self.gt_paths), "Mismatch between NoisyLR and GT counts"

    def __len__(self):
        return len(self.noisy_paths)

    def _normalize(self, noisy, gt):
        # Joint normalization strategy: Ensures physical relative intensities are preserved
        max_val = max(noisy.max(), gt.max())
        if max_val == 0:
            return noisy, gt
        return noisy / max_val, gt / max_val

    def __getitem__(self, idx):
        # Load npy files
        noisy_img = np.load(self.noisy_paths[idx]).astype(np.float32)
        gt_img = np.load(self.gt_paths[idx]).astype(np.float32)

        # Normalize
        noisy_img, gt_img = self._normalize(noisy_img, gt_img)

        # Convert to Tensor (C, H, W) where C=1
        noisy_tensor = torch.from_numpy(noisy_img).unsqueeze(0)
        gt_tensor = torch.from_numpy(gt_img).unsqueeze(0)

        # Data augmentation
        if self.phase == 'train':
            # Random horizontal & vertical flips
            if random.random() > 0.5:
                noisy_tensor = torch.flip(noisy_tensor, [2])
                gt_tensor = torch.flip(gt_tensor, [2])
            if random.random() > 0.5:
                noisy_tensor = torch.flip(noisy_tensor, [1])
                gt_tensor = torch.flip(gt_tensor, [1])
            # Random rotations (0, 90, 180, 270)
            k = random.randint(0, 3)
            if k > 0:
                noisy_tensor = torch.rot90(noisy_tensor, k, [1, 2])
                gt_tensor = torch.rot90(gt_tensor, k, [1, 2])

        return noisy_tensor, gt_tensor


In [ ]:
%%writefile models/adair_semicon.py
import torch
import torch.nn as nn

class AdaIR_SR(nn.Module):
    def __init__(self, pretrained_path=None, in_channels=1, upscale=2):
        super(AdaIR_SR, self).__init__()
        self.upscale = upscale
        
        # For the hackathon, we assume AdaIR backbone expects 3 channels.
        # This wrapper repeats the single channel to 3 channels to use the pretrained backbone safely
        # without randomly reinitializing the first layer weights.
        
        try:
            # Placeholder for actual AdaIR import (this path depends on the cloned repo's structure)
            from adair.basicsr.models.archs.adair_arch import AdaIR
            self.backbone = AdaIR(inp_channels=3, out_channels=3, dim=48, num_blocks=[4, 6, 6, 8], num_refinement_blocks=4)
        except ImportError:
            # Mock backbone for testing scripts if AdaIR isn't downloaded yet
            print("Warning: AdaIR module not found. Using a dummy CNN backbone for structural tests.")
            self.backbone = nn.Sequential(
                nn.Conv2d(3, 48, 3, 1, 1),
                nn.ReLU(inplace=True),
                nn.Conv2d(48, 3, 3, 1, 1)
            )

        # PixelShuffle SR Head
        # Output features from AdaIR -> upscale factor
        # Since input to pixel shuffle needs to be in_channels * (upscale ** 2), which is 1 * 4 = 4
        self.upsampler = nn.Sequential(
            nn.Conv2d(3, in_channels * (upscale ** 2), kernel_size=3, stride=1, padding=1),
            nn.PixelShuffle(upscale)
        )
        
        # Load Pretrained weights
        if pretrained_path:
            self.load_pretrained(pretrained_path)

    def load_pretrained(self, path):
        try:
            state_dict = torch.load(path, map_location='cpu')
            # Extract standard parameter dict if wrapped
            if 'params' in state_dict:
                state_dict = state_dict['params']
            elif 'state_dict' in state_dict:
                state_dict = state_dict['state_dict']
                
            self.backbone.load_state_dict(state_dict, strict=True)
            print(f"Successfully loaded pretrained backbone from {path}")
        except Exception as e:
            print(f"Warning: Could not load pretrained weights from {path}. Error: {e}")

    def forward(self, x):
        # x is [B, 1, 128, 128]
        
        # Grayscale Adapter: project 1 channel to 3 channels for backbone
        x_3c = x.repeat(1, 3, 1, 1)
        
        # Extract features (restored feature rep)
        features = self.backbone(x_3c)
        
        # Upsampling module with PixelShuffle: [B, 3, 128, 128] -> [B, 4, 128, 128] -> [B, 1, 256, 256]
        out = self.upsampler(features)

        # Residual SR skip gives the network a strong interpolation baseline and
        # lets the trainable path focus on denoising and high-frequency detail.
        skip = torch.nn.functional.interpolate(
            x,
            scale_factor=self.upscale,
            mode="bilinear",
            align_corners=False,
        )
        out = out + skip
        
        return out


In [ ]:
%%writefile utils/metrics.py
from typing import Dict, Optional

import numpy as np
import torch
from skimage.metrics import peak_signal_noise_ratio, structural_similarity


def _as_float_image(image: np.ndarray) -> np.ndarray:
    image = np.asarray(image, dtype=np.float32)
    if image.ndim == 3:
        image = image[..., 0]
    return image


def image_metrics(
    prediction: np.ndarray,
    target: np.ndarray,
    lpips_fn: Optional[torch.nn.Module] = None,
    device: Optional[torch.device] = None,
) -> Dict[str, Optional[float]]:
    """Compute restoration metrics for two same-sized grayscale images."""
    prediction = _as_float_image(prediction)
    target = _as_float_image(target)

    if prediction.shape != target.shape:
        raise ValueError(
            f"Prediction and target must have the same shape, got "
            f"{prediction.shape} and {target.shape}."
        )

    data_range = float(max(prediction.max(), target.max()) - min(prediction.min(), target.min()))
    if data_range <= 0:
        data_range = 1.0

    values: Dict[str, Optional[float]] = {
        "psnr": float(peak_signal_noise_ratio(target, prediction, data_range=data_range)),
        "ssim": float(structural_similarity(target, prediction, data_range=data_range)),
        "lpips": None,
    }

    if lpips_fn is None:
        return values

    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    denom = float(max(prediction.max(), target.max()))
    if denom <= 0:
        denom = 1.0

    pred_tensor = torch.from_numpy(prediction / denom).float().unsqueeze(0).unsqueeze(0).to(device)
    target_tensor = torch.from_numpy(target / denom).float().unsqueeze(0).unsqueeze(0).to(device)
    pred_tensor = pred_tensor.repeat(1, 3, 1, 1) * 2 - 1
    target_tensor = target_tensor.repeat(1, 3, 1, 1) * 2 - 1

    with torch.no_grad():
        values["lpips"] = float(lpips_fn(pred_tensor, target_tensor).item())

    return values


In [ ]:
%%writefile utils/restorer.py
import os
from typing import Dict, Optional, Tuple, Union

import numpy as np
import torch
import yaml
from PIL import Image

from models.adair_semicon import AdaIR_SR
from utils.metrics import image_metrics


class ImageRestorer:
    """Load AdaIR and restore noisy low-resolution grayscale images."""

    def __init__(
        self,
        config_path: str = "configs/train.yaml",
        checkpoint_path: Optional[str] = None,
    ):
        with open(config_path, "r") as f:
            self.config = yaml.safe_load(f)

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = AdaIR_SR(
            pretrained_path=self.config["model"].get("pretrained_path"),
            in_channels=self.config["model"]["in_channels"],
            upscale=self.config["model"]["upscale"],
        ).to(self.device)

        ckpt = checkpoint_path or os.environ.get("ADAIR_CHECKPOINT")
        if not ckpt:
            ckpt = os.path.join(
                self.config["training"]["save_dir"],
                f"{self.config['experiment_name']}.pth",
            )

        self.checkpoint_path = ckpt
        self.checkpoint_loaded = False
        if os.path.exists(ckpt):
            self.model.load_state_dict(
                torch.load(ckpt, map_location=self.device)
            )
            self.checkpoint_loaded = True

        self.model.eval()
        self.lpips_fn = None
        try:
            import lpips

            self.lpips_fn = lpips.LPIPS(net="alex").to(self.device).eval()
        except Exception:
            self.lpips_fn = None

    @staticmethod
    def _to_pil(image: Union[Image.Image, np.ndarray]) -> Image.Image:
        if isinstance(image, np.ndarray):
            if image.dtype != np.uint8:
                image = np.clip(image, 0, 255).astype(np.uint8)
            if image.ndim == 2:
                return Image.fromarray(image, mode="L")
            return Image.fromarray(image).convert("L")
        if image.mode != "L":
            return image.convert("L")
        return image

    @staticmethod
    def _to_display_array(gray: np.ndarray) -> np.ndarray:
        gray = gray.astype(np.float32)
        max_val = float(gray.max())
        if max_val > 0:
            gray = gray / max_val
        return (np.clip(gray, 0, 1) * 255).astype(np.uint8)

    def _restore_array(self, noisy: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray, str]:
        noisy = np.asarray(noisy, dtype=np.float32)
        if noisy.ndim == 3:
            noisy = np.squeeze(noisy)
        if noisy.ndim != 2:
            raise ValueError(f"Expected a 2D grayscale .npy array, got shape {noisy.shape}.")

        max_val = float(noisy.max())
        if max_val <= 0:
            raise ValueError("Input array appears to be empty (all zeros).")

        normalized = noisy / max_val
        tensor = (
            torch.from_numpy(normalized)
            .unsqueeze(0)
            .unsqueeze(0)
            .to(self.device)
        )

        with torch.no_grad():
            pred = self.model(tensor)
            pred = torch.clamp(pred, 0, 1).squeeze().cpu().numpy() * max_val

        input_display = self._to_display_array(noisy)
        restored_display = self._to_display_array(pred)

        upscale = self.config["model"]["upscale"]
        status = (
            f"Input: {noisy.shape[1]}x{noisy.shape[0]} -> "
            f"Output: {pred.shape[1]}x{pred.shape[0]} ({upscale}x upscale). "
            f"Device: {self.device}."
        )
        if self.checkpoint_loaded:
            status += f" Checkpoint: {self.checkpoint_path}"
        else:
            status += (
                " Warning: no trained checkpoint found - output uses an untrained model. "
                f"Place weights at `{self.checkpoint_path}` or set ADAIR_CHECKPOINT."
            )

        return input_display, restored_display, pred.astype(np.float32), status

    def restore(
        self, image: Union[Image.Image, np.ndarray]
    ) -> Tuple[np.ndarray, np.ndarray, str]:
        """
        Run restoration on an uploaded image.

        Returns (input_display, restored_display, status_message) as uint8 arrays.
        """
        pil = self._to_pil(image)
        noisy = np.array(pil, dtype=np.float32)
        input_display, restored_display, _, status = self._restore_array(noisy)
        return input_display, restored_display, status

    def restore_with_metrics(
        self,
        image: Union[Image.Image, np.ndarray],
        reference: Optional[Union[Image.Image, np.ndarray]] = None,
    ) -> Tuple[np.ndarray, np.ndarray, str, Dict[str, object]]:
        input_display, restored_display, status = self.restore(image)

        params: Dict[str, object] = {
            "device": str(self.device),
            "checkpoint": self.checkpoint_path if self.checkpoint_loaded else "not loaded",
            "checkpoint_loaded": self.checkpoint_loaded,
            "upscale": self.config["model"]["upscale"],
            "input_size": f"{input_display.shape[1]}x{input_display.shape[0]}",
            "output_size": f"{restored_display.shape[1]}x{restored_display.shape[0]}",
            "psnr": None,
            "ssim": None,
            "lpips": None,
        }

        if reference is None:
            params["metrics_note"] = "Upload a ground-truth/reference image to compute PSNR, SSIM, and LPIPS."
            return input_display, restored_display, status, params

        ref_pil = self._to_pil(reference)
        reference_array = np.array(ref_pil, dtype=np.float32)
        if reference_array.shape != restored_display.shape:
            ref_pil = ref_pil.resize(
                (restored_display.shape[1], restored_display.shape[0]),
                Image.Resampling.BICUBIC,
            )
            reference_array = np.array(ref_pil, dtype=np.float32)
            params["reference_resized"] = True
        else:
            params["reference_resized"] = False

        metrics = image_metrics(
            restored_display.astype(np.float32),
            reference_array,
            lpips_fn=self.lpips_fn,
            device=self.device,
        )
        params.update(metrics)
        if self.lpips_fn is None:
            params["lpips_note"] = "Install lpips to enable LPIPS: pip install lpips"

        return input_display, restored_display, status, params

    def restore_npy(
        self,
        npy_path: str,
        reference_path: Optional[str] = None,
    ) -> Tuple[np.ndarray, np.ndarray, np.ndarray, str, Dict[str, object]]:
        noisy = np.load(npy_path).astype(np.float32)
        input_display, restored_display, restored_array, status = self._restore_array(noisy)

        params: Dict[str, object] = {
            "device": str(self.device),
            "checkpoint": self.checkpoint_path if self.checkpoint_loaded else "not loaded",
            "checkpoint_loaded": self.checkpoint_loaded,
            "upscale": self.config["model"]["upscale"],
            "input_file": os.path.basename(npy_path),
            "input_shape": list(noisy.shape),
            "output_shape": list(restored_array.shape),
            "input_dtype": str(noisy.dtype),
            "output_dtype": str(restored_array.dtype),
            "input_min": float(noisy.min()),
            "input_max": float(noisy.max()),
            "output_min": float(restored_array.min()),
            "output_max": float(restored_array.max()),
            "psnr": None,
            "ssim": None,
            "lpips": None,
        }

        if not reference_path:
            params["metrics_note"] = "Upload a matching ground-truth .npy file to compute PSNR, SSIM, and LPIPS."
            return input_display, restored_display, restored_array, status, params

        reference = np.load(reference_path).astype(np.float32)
        if reference.ndim == 3:
            reference = np.squeeze(reference)
        metrics = image_metrics(
            restored_array,
            reference,
            lpips_fn=self.lpips_fn,
            device=self.device,
        )
        params.update(metrics)
        params["reference_file"] = os.path.basename(reference_path)
        params["reference_shape"] = list(reference.shape)
        if self.lpips_fn is None:
            params["lpips_note"] = "Install lpips to enable LPIPS: pip install lpips"

        return input_display, restored_display, restored_array, status, params


In [ ]:
%%writefile configs/train.yaml
experiment_name: "adair_semicon_baseline"

model:
  name: "AdaIR_SR"
  pretrained_path: "checkpoints/pretrained_adair.pth" # Ensure you download this beforehand
  in_channels: 1
  out_channels: 1
  upscale: 2
  
dataset:
  train_noisy_dir: "train/train/NoisyLR"
  train_gt_dir: "train/train/GT"
  val_noisy_dir: "test/NoisyLR"
  val_gt_dir: "test/GT"
  batch_size: 16
  patch_size: 128
  num_workers: 4
  
training:
  epochs: 200
  learning_rate: 2.0e-4
  optimizer: "AdamW"
  scheduler: "CosineAnnealingLR"
  T_max: 200
  loss: "CharbonnierLoss"
  weight_decay: 1.0e-4
  use_amp: true
  validate_every: 1
  save_dir: "checkpoints/"
  log_dir: "reports/"


In [ ]:
%%writefile train.py
import os
import glob
import yaml
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from datasets.semicon_dataset import SemiconDataset
from models.adair_semicon import AdaIR_SR
from torch.cuda.amp import autocast, GradScaler
import argparse
from tqdm import tqdm
import numpy as np
from utils.metrics import image_metrics

class CharbonnierLoss(nn.Module):
    def __init__(self, eps=1e-3):
        super(CharbonnierLoss, self).__init__()
        self.eps = eps

    def forward(self, x, y):
        diff = x - y
        loss = torch.mean(torch.sqrt((diff * diff) + (self.eps * self.eps)))
        return loss


def evaluate(model, config, device):
    val_noisy_dir = config["dataset"].get("val_noisy_dir")
    val_gt_dir = config["dataset"].get("val_gt_dir") or val_noisy_dir.replace("NoisyLR", "GT")
    noisy_paths = sorted(glob.glob(os.path.join(val_noisy_dir, "*.npy")))
    gt_paths = sorted(glob.glob(os.path.join(val_gt_dir, "*.npy")))

    if not noisy_paths or not gt_paths:
        return None

    totals = {"psnr": 0.0, "ssim": 0.0}
    count = 0
    model.eval()

    with torch.no_grad():
        for noisy_path, gt_path in zip(noisy_paths, gt_paths):
            noisy = np.load(noisy_path).astype(np.float32)
            gt = np.load(gt_path).astype(np.float32)
            max_val = float(max(noisy.max(), gt.max()))
            if max_val <= 0:
                max_val = 1.0

            noisy_tensor = torch.from_numpy(noisy / max_val).unsqueeze(0).unsqueeze(0).to(device)
            pred = torch.clamp(model(noisy_tensor), 0, 1).squeeze().cpu().numpy() * max_val
            values = image_metrics(pred, gt)
            totals["psnr"] += values["psnr"]
            totals["ssim"] += values["ssim"]
            count += 1

    return {name: value / count for name, value in totals.items()}

def train():
    parser = argparse.ArgumentParser()
    parser.add_argument('--config', type=str, default='configs/train.yaml', help='Path to config file')
    args = parser.parse_args()

    with open(args.config, 'r') as f:
        config = yaml.safe_load(f)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    # Dataset & DataLoader
    train_dataset = SemiconDataset(
        noisy_dir=config['dataset']['train_noisy_dir'],
        gt_dir=config['dataset']['train_gt_dir'],
        patch_size=config['dataset']['patch_size'],
        phase='train'
    )
    
    train_loader = DataLoader(
        train_dataset, 
        batch_size=config['dataset']['batch_size'], 
        shuffle=True, 
        num_workers=config['dataset']['num_workers'],
        pin_memory=True
    )

    # Model
    model = AdaIR_SR(
        pretrained_path=config['model'].get('pretrained_path'),
        in_channels=config['model']['in_channels'],
        upscale=config['model']['upscale']
    ).to(device)

    # Loss
    if config['training']['loss'] == 'CharbonnierLoss':
        criterion = CharbonnierLoss().to(device)
    else:
        criterion = nn.L1Loss().to(device) # fallback

    # Optimizer & Scheduler
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=float(config['training']['learning_rate']),
        weight_decay=float(config['training'].get('weight_decay', 0.0)),
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config['training']['T_max'])

    # AMP Scaler
    use_amp = config['training'].get('use_amp', False)
    scaler = GradScaler(enabled=use_amp)

    epochs = config['training']['epochs']
    save_dir = config['training']['save_dir']
    os.makedirs(save_dir, exist_ok=True)
    best_psnr = float("-inf")
    validate_every = int(config["training"].get("validate_every", 1))

    print("Starting Training...")
    for epoch in range(1, epochs + 1):
        model.train()
        epoch_loss = 0.0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{epochs}")
        for noisy_img, gt_img in pbar:
            noisy_img = noisy_img.to(device)
            gt_img = gt_img.to(device)

            optimizer.zero_grad()

            with autocast(enabled=use_amp):
                preds = model(noisy_img)
                loss = criterion(preds, gt_img)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            epoch_loss += loss.item() * noisy_img.size(0)
            pbar.set_postfix({'loss': loss.item()})

        scheduler.step()
        
        avg_loss = epoch_loss / len(train_dataset)
        print(f"Epoch {epoch} finished. Average Loss: {avg_loss:.6f}")

        if validate_every > 0 and epoch % validate_every == 0:
            metrics = evaluate(model, config, device)
            if metrics:
                print(f"Validation PSNR: {metrics['psnr']:.4f} SSIM: {metrics['ssim']:.4f}")
                if metrics["psnr"] > best_psnr:
                    best_psnr = metrics["psnr"]
                    torch.save(model.state_dict(), os.path.join(save_dir, f"{config['experiment_name']}_best.pth"))
                    torch.save(model.state_dict(), os.path.join(save_dir, f"{config['experiment_name']}.pth"))
                    print(f"Saved new best checkpoint with PSNR {best_psnr:.4f}")

        if epoch % 10 == 0 or epoch == epochs:
            # Save checkpoint
            save_path = os.path.join(save_dir, f"{config['experiment_name']}_epoch_{epoch}.pth")
            torch.save(model.state_dict(), save_path)
            if best_psnr == float("-inf"):
                torch.save(model.state_dict(), os.path.join(save_dir, f"{config['experiment_name']}.pth"))

if __name__ == '__main__':
    train()


In [ ]:
%%writefile validate.py
import os
import argparse
import yaml
import glob
import numpy as np
import torch
from models.adair_semicon import AdaIR_SR
from utils.metrics import image_metrics
try:
    import lpips
except ImportError:
    lpips = None

def validate():
    parser = argparse.ArgumentParser()
    parser.add_argument('--config', type=str, default='configs/train.yaml')
    parser.add_argument('--checkpoint', type=str, required=True, help="Path to weights")
    args = parser.parse_args()

    with open(args.config, 'r') as f:
        config = yaml.safe_load(f)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Initialize LPIPS
    if lpips is not None:
        lpips_fn = lpips.LPIPS(net='alex').to(device)
    else:
        print("LPIPS package not found. Run pip install lpips")
        lpips_fn = None

    model = AdaIR_SR(
        pretrained_path=config['model'].get('pretrained_path'),
        in_channels=config['model']['in_channels'],
        upscale=config['model']['upscale']
    ).to(device)
    
    # Load state dict
    model.load_state_dict(torch.load(args.checkpoint, map_location=device))
    model.eval()

    # Metrics
    psnr_total = 0.0
    ssim_total = 0.0
    lpips_total = 0.0
    
    # Note: Requires GT files to exist in test folder for full validation
    val_noisy_dir = config['dataset']['val_noisy_dir']
    val_gt_dir = config['dataset'].get('val_gt_dir') or val_noisy_dir.replace("NoisyLR", "GT")
    
    noisy_paths = sorted(glob.glob(os.path.join(val_noisy_dir, '*.npy')))
    gt_paths = sorted(glob.glob(os.path.join(val_gt_dir, '*.npy')))
    
    if len(gt_paths) == 0:
        print("Warning: No GT files found for validation. Evaluation needs GT pairs.")
        return

    count = 0
    with torch.no_grad():
        for n_path, g_path in zip(noisy_paths, gt_paths):
            noisy = np.load(n_path).astype(np.float32)
            gt = np.load(g_path).astype(np.float32)

            max_val = max(noisy.max(), gt.max())
            noisy_norm = noisy / max_val if max_val > 0 else noisy
            
            n_tensor = torch.from_numpy(noisy_norm).unsqueeze(0).unsqueeze(0).to(device)
            
            pred_tensor = model(n_tensor)
            
            # Clamp directly after inference
            pred_tensor = torch.clamp(pred_tensor, 0, 1)
            pred = pred_tensor.squeeze().cpu().numpy() * max_val
            
            metrics = image_metrics(pred, gt, lpips_fn=lpips_fn, device=device)
            psnr_val = metrics["psnr"]
            ssim_val = metrics["ssim"]
            
            psnr_total += psnr_val
            ssim_total += ssim_val
            
            # LPIPS expects inputs from [-1, 1], shape N C H W
            if lpips_fn:
                lpips_total += metrics["lpips"]
            
            count += 1

    print(f"Validation Results over {count} images:")
    print(f"PSNR:  {psnr_total / count:.4f}")
    print(f"SSIM:  {ssim_total / count:.4f}")
    if lpips_fn:
        print(f"LPIPS: {lpips_total / count:.4f}")

if __name__ == '__main__':
    validate()


In [ ]:
%%writefile inference.py
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
import argparse
import glob
from models.adair_semicon import AdaIR_SR
import yaml
from utils.metrics import image_metrics

def infer():
    parser = argparse.ArgumentParser()
    parser.add_argument('--config', type=str, default='configs/train.yaml')
    parser.add_argument('--checkpoint', type=str, required=True)
    parser.add_argument('--input', type=str, help='Path to specific NoisyLR .npy file')
    parser.add_argument('--output-dir', type=str, default='results')
    parser.add_argument('--limit', type=int, default=0, help='Number of validation files to process when --input is not set. 0 means all files.')
    args = parser.parse_args()

    with open(args.config, 'r') as f:
        config = yaml.safe_load(f)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = AdaIR_SR(
        in_channels=config['model']['in_channels'],
        upscale=config['model']['upscale']
    ).to(device)
    model.load_state_dict(torch.load(args.checkpoint, map_location=device))
    model.eval()
    
    os.makedirs(args.output_dir, exist_ok=True)

    def process_file(n_path):
        noisy = np.load(n_path).astype(np.float32)
        gt_path = n_path.replace("NoisyLR", "GT")
        has_gt = os.path.exists(gt_path)
        
        gt = np.load(gt_path).astype(np.float32) if has_gt else None
        max_val = noisy.max()
        if has_gt: max_val = max(max_val, gt.max())
            
        n_input = (noisy / max_val) if max_val > 0 else noisy
        n_tensor = torch.from_numpy(n_input).unsqueeze(0).unsqueeze(0).to(device)
        
        with torch.no_grad():
            pred = model(n_tensor)
            pred = torch.clamp(pred, 0, 1).squeeze().cpu().numpy() * max_val
            
        stem = os.path.splitext(os.path.basename(n_path))[0]
        np.save(os.path.join(args.output_dir, f'{stem}_restored.npy'), pred.astype(np.float32))

        metrics_text = ''
        if has_gt:
            metrics = image_metrics(pred, gt)
            metrics_text = (
                f" | PSNR: {metrics['psnr']:.4f} "
                f"SSIM: {metrics['ssim']:.4f}"
            )

        # Visualization
        fig, axes = plt.subplots(1, 3 if has_gt else 2, figsize=(15, 5))
        axes[0].imshow(noisy, cmap='gray')
        axes[0].set_title('Input (Noisy 128x128)')
        
        axes[1].imshow(pred, cmap='gray')
        axes[1].set_title('Prediction (256x256)')
        
        if has_gt:
            axes[2].imshow(gt, cmap='gray')
            axes[2].set_title('Ground Truth (256x256)')
        for ax in np.ravel(axes):
            ax.axis('off')

        fig.savefig(os.path.join(args.output_dir, f'{stem}_comparison.png'), bbox_inches='tight')
        plt.close(fig)
        print(f"Saved {stem}_restored.npy and {stem}_comparison.png{metrics_text}")

    if args.input:
        process_file(args.input)
    else:
        # Process a random file from validation set
        val_noisy_dir = config['dataset']['val_noisy_dir']
        files = glob.glob(os.path.join(val_noisy_dir, '*.npy'))
        if args.limit > 0:
            files = files[:args.limit]
        if files:
            for file_path in files:
                process_file(file_path)
        else:
            print("No test files found to infer.")

if __name__ == '__main__':
    infer()


## 4. Upload or Mount Dataset

Expected folder layout:

```text
train/train/NoisyLR/*.npy
train/train/GT/*.npy
test/NoisyLR/*.npy
test/GT/*.npy
```

Option A: upload a zip named `semicon_data.zip` to Colab Files and unzip it. Option B: mount Google Drive and copy/symlink your folders into the same layout.


In [ ]:
# Option A: unzip uploaded dataset archive if present.
import os
if os.path.exists('/content/semicon_data.zip'):
    !unzip -q -o /content/semicon_data.zip -d /content
else:
    print('Upload /content/semicon_data.zip or mount Drive in the next cell.')


In [ ]:
# Option B: mount Drive, then adjust these paths if your dataset is stored there.
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r /content/drive/MyDrive/semicon/train /content/train
# !cp -r /content/drive/MyDrive/semicon/test /content/test


## 5. Configure Paths and Hyperparameters

Edit these values if your folders differ. The default training run saves the best validation checkpoint as `checkpoints/adair_semicon_baseline.pth`.


In [ ]:
import yaml

config_path = 'configs/train.yaml'
with open(config_path, 'r') as f:
    cfg = yaml.safe_load(f)

cfg['dataset']['train_noisy_dir'] = 'train/train/NoisyLR'
cfg['dataset']['train_gt_dir'] = 'train/train/GT'
cfg['dataset']['val_noisy_dir'] = 'test/NoisyLR'
cfg['dataset']['val_gt_dir'] = 'test/GT'
cfg['dataset']['batch_size'] = 16
cfg['training']['epochs'] = 200
cfg['training']['learning_rate'] = 2.0e-4
cfg['training']['validate_every'] = 1

with open(config_path, 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

cfg


## 6. Sanity Check Dataset


In [ ]:
import glob, os, numpy as np

for name, path in {
    'train noisy': 'train/train/NoisyLR/*.npy',
    'train gt': 'train/train/GT/*.npy',
    'test noisy': 'test/NoisyLR/*.npy',
    'test gt': 'test/GT/*.npy',
}.items():
    files = sorted(glob.glob(path))
    print(f'{name}: {len(files)} files')
    if files:
        arr = np.load(files[0])
        print(' ', os.path.basename(files[0]), arr.shape, arr.dtype, float(arr.min()), float(arr.max()))


## 7. Train Model

This trains and saves the best PSNR checkpoint. If you already have a checkpoint, skip this cell and upload/copy it to `checkpoints/adair_semicon_baseline.pth`.


In [ ]:
!python train.py --config configs/train.yaml


## 8. Validate Metrics


In [ ]:
!python validate.py --config configs/train.yaml --checkpoint checkpoints/adair_semicon_baseline.pth


## 9. Test Outputs on `.npy` Files

This restores test inputs, saves restored `.npy` files, and writes comparison PNGs into `results/`.


In [ ]:
!python inference.py --config configs/train.yaml --checkpoint checkpoints/adair_semicon_baseline.pth --output-dir results --limit 10


## 10. Preview Results


In [ ]:
import glob
from IPython.display import Image, display

for image_path in sorted(glob.glob('results/*_comparison.png'))[:5]:
    print(image_path)
    display(Image(filename=image_path))


## 11. Download Checkpoint and Outputs


In [ ]:
!zip -q -r adair_results.zip checkpoints results reports
from google.colab import files
files.download('adair_results.zip')
